# Milestone 4 — M4D: diffuse camera sampling

Sample nodal `Φ` at camera **hit points** → `I_diffuse`.

Plan: [`plans/milestone_04/04_diffusion_plan.md`](../../plans/milestone_04/04_diffusion_plan.md). Previous: `04c`. Next: `04e`.

## Limitations

- Requires exact `hit_points` (face indices alone are insufficient).
- This notebook uses `interpolate=False` for M4 historical parity; production defaults to barycentric interpolation.


In [ ]:
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent

!pip install --quiet --no-cache-dir "{ROOT}[fem]" -c "{ROOT}/requirements.txt"

from gummybear.paths import display_path

print(f"ROOT={{display_path(ROOT)}}")



In [6]:
import numpy as np

from gummybear.geometry import inspect_stl
from gummybear.optics import (
    OpticalMaterialConfig,
    deposit_ray_source,
    generate_diffusion_mesh,
    sample_diffuse_image,
    solve_diffusion,
)
from gummybear.rays import PinholeCameraConfig, first_visible_hits_with_points, make_camera_rays
from gummybear_validation.helpers import assert_live_netgen_mesh, make_centroid_axis_ray
from gummybear_validation.plotting import plot_camera_scalar


In [7]:
inspection = inspect_stl(ROOT / "cad" / "proto_bear_head.stl")
mesh = inspection["mesh"]
diff_mesh = generate_diffusion_mesh(mesh, target_elements=2000)
assert_live_netgen_mesh(diff_mesh)

material = OpticalMaterialConfig(mu_scatter=0.2, mu_absorption=0.001)
ray, _p0, _p1 = make_centroid_axis_ray(diff_mesh, axis="x", intensity=1.0)
dep = deposit_ray_source(diff_mesh, ray, mu_s=material.mu_s, mu_a=material.mu_a)

solved = solve_diffusion(
    diff_mesh,
    S_clean=dep.S_clean,
    D=material.diffusion_coefficient,
    mu_a=material.mu_a,
    extrapolation_length=5.0,
)

camera = PinholeCameraConfig(
    camera_position=(0.0, -60.0, 0.0),
    look_at=(0.0, 0.0, 10.0),
    up=(0.0, 0.0, 1.0),
    fov_deg=35.0,
    resolution=256,
)
cam_rays = make_camera_rays(camera)
cam_valid, _depth, _faces, cam_points = first_visible_hits_with_points(mesh, cam_rays)
H, W = cam_rays.sample_shape


In [ ]:
diffuse = sample_diffuse_image(
    diff_mesh,
    solved.Phi_nodes,
    hit_points=cam_points,
    valid_mask=cam_valid,
    sample_shape=(H, W),
    interpolate=False,
)

camera_mask = cam_valid.reshape(H, W)
assert np.all(diffuse.I_diffuse[~camera_mask] == 0.0)
plot_camera_scalar(diffuse.I_diffuse, title="I_diffuse (interpolate=False)")
